# Lezione 12 - Ingegneria dei dati: formati di file e processo ETL

**Obiettivi della lezione**

- Capire cos'è il processo **ETL** (Estrazione, Trasformazione, Caricamento) e perché è il flusso mentale dietro quasi ogni lavoro con i dati.
- Leggere e scrivere file nei formati più comuni: **CSV**, **JSON**, **XML** e (opzionalmente) **XLSX**.
- Applicare una prima esplorazione di un DataFrame con `.info()`, `.describe()`, `.isnull()` e `.dtypes`.

Tutto il codice di questo notebook è autonomo: non dipende dallo scaricare nessun dataset esterno. Creeremo i nostri dati fittizi (tipo "dataset medico") con NumPy e Pandas, usando un seed fisso in modo che il risultato sia sempre lo stesso.

## 1. Il processo ETL

Ogni volta che lavori con i dati, in fondo stai facendo tre cose:

1. **Estrazione (Extract)**: procurarsi i dati da dove si trovano (un file CSV, un JSON di una API, un Excel che ti hanno mandato per email, ecc.). In questo passo non tocchi ancora il contenuto, lo porti solo nel tuo ambiente di lavoro.
2. **Trasformazione (Transform)**: pulire e unificare. Eliminare ciò che non serve, convertire i tipi di dato, uniformare i formati tra fonti diverse.
3. **Caricamento (Load)**: lasciare i dati già trasformati pronti per l'analisi - nel tuo caso, quasi sempre un DataFrame di Pandas.

Ogni volta che fai `df = pd.read_csv(...)` seguito da `df.dropna()` o dal rinominare colonne, stai già facendo ETL su piccola scala. Il resto di questa lezione è, in fondo, un catalogo di tecniche per il passo di "Estrazione" a seconda del formato in cui arrivano i dati.

## 2. Formato CSV

Il **CSV** (comma-separated values) è testo semplice tabulare: ogni riga è un record, e i campi sono separati da virgole. Conosci già `pd.read_csv()` e `df.to_csv()`; in questa sezione ripassiamo due dettagli utili nella pratica e lavoriamo con un dataset nostro.

Prima creiamo un DataFrame fittizio tipo "dataset clinico" con NumPy, usando un seed fisso (`np.random.seed`) in modo che i risultati siano riproducibili.

In [ ]:
import numpy as np
import pandas as pd

np.random.seed(42)
n = 15

pacientes = pd.DataFrame({
    "id_paciente": range(1, n + 1),
    "embarazos": np.random.randint(0, 6, size=n),
    "glucosa": np.random.randint(80, 200, size=n).astype(float),
    "presion_arterial": np.random.randint(60, 100, size=n),
    "imc": np.round(np.random.uniform(18.5, 40.0, size=n), 1),
    "edad": np.random.randint(21, 65, size=n),
})

# Inseriamo apposta un paio di valori mancanti per esercitarci con .isnull() più avanti
pacientes.loc[3, "glucosa"] = np.nan
pacientes.loc[7, "imc"] = np.nan

pacientes.head()

### Quando il CSV non ha intestazioni

Se un CSV non include una riga di intestazione, `read_csv` prende di default la prima riga di dati come se fossero i nomi delle colonne - un errore classico. Si risolve con `header=None` e assegnando tu stesso i nomi dopo, usando l'attributo `.columns`.

In [ ]:
# Simuliamo un CSV "senza intestazioni" scrivendolo noi stessi
direcciones_sin_encabezado = pd.DataFrame([
    ["Rita", "Perez", "Av. Libertad 100", "Cordoba", "Cordoba", "5000"],
    ["Juan", "Gomez", "Calle Falsa 123", "Rosario", "Santa Fe", "2000"],
])
direcciones_sin_encabezado.to_csv("direcciones.csv", index=False, header=False)

# Leggendolo con header=None, Pandas non assume che la prima riga siano nomi di colonna
df_direcciones = pd.read_csv("direcciones.csv", header=None)
df_direcciones.columns = ["Nombre", "Apellido", "Calle", "Ciudad", "Provincia", "Codigo_Postal"]
df_direcciones

### La funzione `.transform()`

`.transform()` applica una funzione a ogni elemento di un DataFrame e restituisce un nuovo DataFrame con il risultato, senza bisogno di scrivere un ciclo. È tipica del passo di "Trasformazione" dell'ETL quando devi applicare la stessa operazione a un'intera tabella numerica in un colpo solo.

In [ ]:
df_ejemplo = pd.DataFrame(np.array([[1, 2, 3], [4, 5, 6], [7, 8, 9]]), columns=["a", "b", "c"])

# Sommare 10 a ogni elemento
df_mas_10 = df_ejemplo.transform(func=lambda x: x + 10)
print(df_mas_10)

# Applicare una funzione per nome invece di una lambda
df_sqrt = df_ejemplo.transform(func=["sqrt"])
print(df_sqrt)

### Salvare e rileggere il CSV (il passo di "Caricamento")

`index=False` evita che Pandas aggiunga una colonna extra con il numero di riga - senza questo argomento, ogni volta che salvi e rileggi il file accumuli una colonna di indice in più.

In [ ]:
pacientes.to_csv("pacientes.csv", index=False)

pacientes_desde_csv = pd.read_csv("pacientes.csv")
pacientes_desde_csv.head()

## 3. Formato JSON

**JSON** conserva coppie chiave-valore e liste, proprio come un dizionario di Python. Convertire un oggetto Python in JSON si chiama **serializzare**; il processo inverso si chiama **deserializzare**. Python porta tutto questo nel pacchetto integrato `json`, senza installare nulla.

- `json.dump(oggetto, file)` serializza direttamente in un file aperto.
- `json.dumps(oggetto)` serializza in una stringa in memoria (utile per stampare o inviare tramite un altro canale).
- `json.load(file)` deserializza da un file aperto.
- `json.loads(stringa)` deserializza da una stringa in memoria.

Memorizza la coppia: `dump`/`dumps` vanno da Python verso JSON, `load`/`loads` vanno da JSON verso Python.

In [ ]:
import json

paciente_dict = {
    "id_paciente": 1,
    "nombre": "Ana",
    "edad": 34,
    "mediciones": {"glucosa": 105, "imc": 24.3},
}

# Serializzare direttamente su file con dump()
with open("paciente.json", "w") as f:
    json.dump(paciente_dict, f, indent=4)

# Serializzare in stringa con dumps() -utile per stampare con formattazione leggibile
paciente_json_str = json.dumps(paciente_dict, indent=4)
print(paciente_json_str)

In [ ]:
# Deserializzare da file con load()
with open("paciente.json", "r") as f:
    paciente_cargado = json.load(f)

print(paciente_cargado)
print(type(paciente_cargado))

# Deserializzare da stringa con loads()
paciente_desde_string = json.loads(paciente_json_str)
print(paciente_desde_string == paciente_cargado)

Pandas ha anche le proprie funzioni `pd.read_json()` / `df.to_json()`, che seguono lo stesso schema di `read_csv`/`to_csv` ma per JSON.

In [ ]:
pacientes.to_json("pacientes.json", orient="records", indent=2)

pacientes_desde_json = pd.read_json("pacientes.json")
pacientes_desde_json.head()

## 4. Formato XLSX (opzionale)

**XLSX** è il formato di Excel. Il flusso è identico a quello del CSV, solo con `pd.read_excel()` / `df.to_excel()`. Perché Pandas possa leggere e scrivere XLSX serve la libreria `openpyxl` installata sotto il cofano.

> **Nota:** questa sezione rileva se `openpyxl` è disponibile nell'ambiente. Se non lo è, il codice non fallisce: stampa semplicemente un avviso e mostra quale sarebbe lo schema di codice da usare.

In [ ]:
try:
    import openpyxl  # noqa: F401

    pacientes.to_excel("pacientes.xlsx", index=False)
    pacientes_desde_excel = pd.read_excel("pacientes.xlsx")
    print("openpyxl è disponibile: pacientes.xlsx è stato scritto e letto correttamente.")
    print(pacientes_desde_excel.head())
except ImportError:
    print("openpyxl non è installato in questo ambiente.")
    print("Per lavorare con XLSX installa la libreria con: pip install openpyxl")
    print("Lo schema di codice, una volta installato, è:")
    print('  pacientes.to_excel("pacientes.xlsx", index=False)')
    print('  pd.read_excel("pacientes.xlsx")')

## 5. Formato XML

Pandas non porta di serie un modo completo per leggere e scrivere XML. Per questo usiamo il modulo integrato `xml.etree.ElementTree`, che rappresenta il documento XML come un albero di nodi che si può percorrere.

### Scrivere un XML da zero

In [ ]:
import xml.etree.ElementTree as ET

pacientes_xml = ET.Element("pacientes")
for _, fila in pacientes.head(3).iterrows():
    paciente_el = ET.SubElement(pacientes_xml, "paciente")
    ET.SubElement(paciente_el, "id").text = str(fila["id_paciente"])
    ET.SubElement(paciente_el, "edad").text = str(fila["edad"])
    ET.SubElement(paciente_el, "imc").text = str(fila["imc"])

arbol = ET.ElementTree(pacientes_xml)
arbol.write("pacientes.xml")
print("File pacientes.xml creato")

### Leggere un XML manualmente con ElementTree

Lo schema è: fare il parsing del file, definire le colonne attese, e percorrere ogni nodo estraendo il testo di ogni sottoelemento.

In [ ]:
import xml.etree.ElementTree as etree

tree = etree.parse("pacientes.xml")
root = tree.getroot()
columnas = ["id", "edad", "imc"]

filas = []
for nodo in root:
    filas.append({
        "id": nodo.find("id").text,
        "edad": nodo.find("edad").text,
        "imc": nodo.find("imc").text,
    })

pacientes_desde_xml = pd.DataFrame(filas, columns=columnas)
pacientes_desde_xml

### Leggere un XML con `pd.read_xml()`

Se la struttura dell'XML è semplice e ripetitiva, `pd.read_xml()` risparmia il percorso manuale: gli passi uno `xpath` che indica quali nodi diventano righe.

Per default, `pd.read_xml()` usa il parser `lxml` (una libreria esterna che va installata a parte). Siccome in questo notebook vogliamo dipendere solo dalla libreria standard, passiamo `parser="etree"` perché usi `xml.etree.ElementTree` sotto il cofano - in quel caso lo `xpath` si scrive in forma relativa (`"./paciente"`).

In [ ]:
pacientes_desde_xml_pd = pd.read_xml("pacientes.xml", xpath="./paciente", parser="etree")
pacientes_desde_xml_pd

## 6. Prima esplorazione di un dataset

Con l'ETL e i formati coperti, chiudiamo applicando gli strumenti di esplorazione di Pandas sul nostro DataFrame `pacientes`. Questa fase è, in fondo, la validazione che l'ETL sia andato bene: se hai appena caricato un dataset, la prima cosa da fare è verificare che il caricamento abbia portato quello che ti aspettavi.

### `.info()`: riepilogo strutturale

In [ ]:
pacientes.info()

`.info()` mostra quante righe ha il DataFrame, il nome e il tipo di ogni colonna, e quanti valori non nulli ci sono in ciascuna. È il tuo primo controllo di salute: se una colonna che ti aspettavi numerica appare come `object`, o il conteggio dei non nulli è inferiore al totale delle righe, sai già dove guardare.

### `.describe()`: statistica descrittiva

In [ ]:
pacientes.describe()

`.describe()` dà statistica descrittiva delle colonne numeriche: conteggio, media, deviazione standard, minimo, massimo e i percentili 25/50/75.

### `.isnull()`: rilevamento di valori mancanti

In [ ]:
missing_data = pacientes.isnull()
missing_data.head()

In [ ]:
# Contare quanti True (valori mancanti) ci sono in ogni colonna
for column in missing_data.columns.values.tolist():
    print(column)
    print(missing_data[column].value_counts())
    print("")

### `.dtypes`: tipo di dato di ogni colonna

In [ ]:
pacientes.dtypes

Se tutto questo risulta come ti aspettavi - tipi corretti, valori mancanti identificati e spiegati - hai già la conferma che il passo di "Estrazione e Trasformazione" dell'ETL è andato a buon fine e sei pronto a passare all'analisi.

## Riepilogo della lezione

- **ETL** è Estrazione, Trasformazione e Caricamento: il flusso mentale dietro quasi ogni lavoro con i dati.
- **CSV**: testo semplice tabulare. `pd.read_csv()` (con `header=None` se non ha intestazioni) / `df.to_csv(..., index=False)`.
- **JSON**: coppie chiave-valore e liste, proprio come un dizionario di Python. Serializzare con `json.dump()`/`json.dumps()`, deserializzare con `json.load()`/`json.loads()`. Esistono anche `pd.read_json()`/`df.to_json()`.
- **XLSX**: fogli di calcolo di Excel. `pd.read_excel()`/`df.to_excel()`, richiedono `openpyxl` installato.
- **XML**: albero di nodi. Si percorre a mano con `xml.etree.ElementTree`, oppure si usa `pd.read_xml(file, xpath=...)` se la struttura è semplice.
- `.info()`, `.describe()`, `.isnull()` e `.dtypes` sono il tuo primo controllo di salute di qualsiasi dataset appena caricato.